# Fase 4 · M04: Distribuciones Categóricas

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 4 — EDA Final |
| **Módulo** | M04 — Distribuciones Categóricas |

---

## 🎯 Qué hace

Análisis univariante de las variables categóricas del dataset (número detectado dinámicamente): frecuencias, proporciones, entropía, categorías raras y análisis de vía de acceso.

## 📋 Requisitos

- `data/04_eda/df_eda_final.parquet`

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `docs/html/fase4/m04_distribuciones_cat.html` | Informe de distribuciones categóricas |

## 🔄 Flujo

```
data/04_eda/df_eda_final.parquet
    ↓ Frecuencias y proporciones
    ↓ Análisis de categorías raras
    → docs/html/fase4/m04_distribuciones_cat.html
```

## ➡️ Siguiente

`f4_m05_bivariante.ipynb` — análisis bivariante vs target


In [1]:
# ── CSS + JS para botones Interpretar ────────────────────────────────────────
js_css_html = (
    '<style>'
    '.ibt{display:inline-flex;align-items:center;gap:5px;padding:5px 12px;'
    'background:#3182ce;color:#fff;border:none;border-radius:5px;'
    'font-size:12px;font-weight:600;cursor:pointer;float:right;margin-top:-2px;}'
    '.ibt:hover{background:#2b6cb0;}'
    '.ipn{display:none;background:#f7fafc;border:1px solid #e2e8f0;'
    'border-radius:6px;padding:14px 16px;margin-top:8px;font-size:13px;'
    'color:#2d3748;line-height:1.6;}'
    '.ipn.vis{display:block;}'
    '.tg{background:#c6f6d5;color:#276749;padding:1px 6px;border-radius:3px;font-weight:600;}'
    '.tb{background:#fed7d7;color:#9b2c2c;padding:1px 6px;border-radius:3px;font-weight:600;}'
    '.tm{background:#fefcbf;color:#744210;padding:1px 6px;border-radius:3px;font-weight:600;}'
    '</style>'
    '<script>'
    'function tg(id){'
    'var p=document.getElementById(id);'
    'var b=document.querySelector("[data-pid=\'" + id + "\']");'
    'if(!p||!b)return;'
    'var vis=p.classList.toggle("vis");'
    'b.textContent=vis?"✖ Cerrar":"📖 Interpretar";}'
    '</script>'
)

def h2btn(titulo, pid):
    return (
        f'<div style="display:flex;align-items:center;justify-content:space-between;'
        f'margin:28px 0 8px;">'
        f'<h2 style="font-size:18px;font-weight:700;color:#1a202c;margin:0;">{titulo}</h2>'
        f'<button class="ibt" data-pid="{pid}" onclick="tg(\'{pid}\')">' 
        f'📖 Interpretar</button></div>'
    )

def panel(pid, texto):
    return f'<div id="{pid}" class="ipn">{texto}</div>'


In [2]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================

import sys
import warnings
import base64
import io
from pathlib import Path

warnings.filterwarnings('ignore')

ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent


sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pywaffle import Waffle
import altair as alt
import pygal
from pygal.style import CleanStyle
import plotly.express as px
import plotly.graph_objects as go
import folium
import math

from src.config import (RUTA_HTML, ETIQUETAS_VARIABLES, DICCIONARIO_RAMAS,
                        UNIVERSIDAD_ORIGEN_NOMBRES, info_entorno)
from src.utils import crear_directorios, formato_numero_es

# ── Paneles de interpretación dinámica ──────────────
panel_categoricas = panel('categoricas', 'Variables de alta cardinalidad (universidad_origen, provincia) tienen categorías con pocas observaciones. Categorías raras (<1%) se agruparán en \"Otros\" antes del modelado para evitar sobreajuste en árboles con poca data por hoja.')
panel_chi2 = panel('chi2', 'p-value < 0.05 = asociación estadísticamente significativa. V de Cramér: >0.3 <span class=\"tg\">fuerte</span> · 0.1–0.3 <span class=\"tm\">moderada</span> · <0.1 <span class=\"tb\">débil</span>. Con 33k registros, casi todo es significativo — la V de Cramér es el indicador real de importancia.')

from src.html import (generar_kpis_html, generar_seccion_html,
                      generar_html_navegacion_completa, guardar_html)
from src.html.render import render_base_html

RUTA_EDA = ROOT / 'data' / '04_eda'
RUTA_FASE4_HTML = RUTA_HTML / 'fase4'
RUTA_GRAFICOS = RUTA_FASE4_HTML / 'graficos'
crear_directorios([RUTA_FASE4_HTML, RUTA_GRAFICOS])

fmt = formato_numero_es

def etiq(col):
    return ETIQUETAS_VARIABLES.get(col, col)


def fig_a_base64(fig, dpi=150):
    """Convierte figura Matplotlib a base64."""
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=dpi, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.close(fig)
    buf.seek(0)
    return base64.b64encode(buf.read()).decode('utf-8')


def img_html(b64, ancho='100%'):
    return (f'<img src="data:image/png;base64,{b64}" '
            f'style="width:{ancho};max-width:100%;display:block;margin:0 auto;">')


def guardar_altair(chart, nombre):
    """Guarda gráfico Altair como HTML y devuelve iframe."""
    ruta = RUTA_GRAFICOS / f'{nombre}.html'
    chart.save(str(ruta))
    return f'<iframe src="graficos/{nombre}.html" width="100%" height="500" frameborder="0"></iframe>'


def guardar_plotly(fig, nombre, height=500):
    """Guarda gráfico Plotly como HTML y devuelve iframe."""
    ruta = RUTA_GRAFICOS / f'{nombre}.html'
    fig.write_html(str(ruta), include_plotlyjs='cdn',
                   config={'displayModeBar': True, 'displaylogo': False})
    return f'<iframe src="graficos/{nombre}.html" width="100%" height="{height}" frameborder="0"></iframe>'


# Paleta coherente
COLOR_PRINCIPAL = '#3182ce'
COLOR_ALERTA = '#ed8936'
COLOR_ERROR = '#e53e3e'
COLOR_OK = '#38a169'
COLOR_MUJER = '#e53e3e'
COLOR_HOMBRE = '#3182ce'

info_entorno()


✓ Directorios verificados: 2
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [3]:
# ============================================================================
# CELDA 2: CARGA DE DATOS Y CLASIFICACIÓN
# ============================================================================

df = pd.read_parquet(RUTA_EDA / 'df_eda_final.parquet')
n_total = len(df)
print(f'Dataset: {n_total:,} filas × {len(df.columns)} columnas')

# Categóricas (str)
categoricas = [c for c in df.columns
               if c != 'abandono' and df[c].dtype in ['object', 'string']]

# Clasificar por cardinalidad
baja_card = [c for c in categoricas if df[c].nunique() <= 5]
media_card = [c for c in categoricas if 6 <= df[c].nunique() <= 15]
alta_card  = [c for c in categoricas if df[c].nunique() > 15]

n_con_nulos = sum(1 for c in categoricas if df[c].isna().sum() > 0)

print(f'\nCategóricas: {len(categoricas)}')
print(f'  Baja (≤5):   {baja_card}')
print(f'  Media (6-15): {media_card}')
print(f'  Alta (>15):   {alta_card}')
print(f'  Con nulos: {n_con_nulos}')

# Resumen para tabla de dominancia
resumen = []
for c in categoricas:
    vc = df[c].value_counts()
    moda = vc.index[0]
    pct_moda = vc.iloc[0] / df[c].notna().sum() * 100
    nulos = df[c].isna().sum()
    # Entropía de Shannon
    probs = vc / vc.sum()
    entropia = -(probs * np.log2(probs)).sum()
    max_entropia = np.log2(df[c].nunique())
    resumen.append({
        'Variable': etiq(c),
        'Categorías': df[c].nunique(),
        'Nulos': nulos,
        '% Nulos': round(nulos / n_total * 100, 1),
        'Moda': moda[:35],
        '% Moda': round(pct_moda, 1),
        'Entropía': round(entropia, 2),
        'Entropía máx.': round(max_entropia, 2),
        'Ratio entropía': round(entropia / max_entropia, 2) if max_entropia > 0 else 0,
    })

df_resumen = pd.DataFrame(resumen).sort_values('Categorías')
print('\nResumen de dominancia:')
print(df_resumen.to_string(index=False))


# ============================================================================
# ANÁLISIS ADICIONALES — Chi², encoding, nulos, tasa abandono
# ============================================================================
from scipy.stats import chi2_contingency

# --- Chi² + V de Cramér por variable vs abandono ---
chi2_results = []
for c in categoricas:
    tabla = pd.crosstab(df[c].fillna('_nulo_'), df['abandono'])
    chi2, p, dof, _ = chi2_contingency(tabla)
    n = tabla.values.sum()
    k = min(tabla.shape) - 1
    v = round(float((chi2 / (n * k)) ** 0.5), 4) if n * k > 0 else 0
    tasa_max = df.groupby(c)['abandono'].mean().max() * 100
    tasa_min = df.groupby(c)['abandono'].mean().min() * 100
    chi2_results.append({
        'variable': c,
        'etiqueta': etiq(c),
        'chi2': round(chi2, 1),
        'p_valor': round(p, 6),
        'cramer_v': v,
        'magnitud': 'Fuerte' if v >= 0.3 else 'Moderada' if v >= 0.1 else 'Débil',
        'tasa_max_pct': round(tasa_max, 1),
        'tasa_min_pct': round(tasa_min, 1),
        'rango_tasas': round(tasa_max - tasa_min, 1),
        'significativa': p < 0.05,
    })

df_chi2 = pd.DataFrame(chi2_results).sort_values('cramer_v', ascending=False)
print("\nChi² + V de Cramér vs abandono:")
print(df_chi2[['etiqueta','cramer_v','magnitud','rango_tasas','p_valor']].to_string(index=False))

# --- Encoding recomendado por variable ---
ENCODING_REGLAS = {
    # Baja cardinalidad (≤5) → OneHotEncoder para lineales, OrdinalEncoder para árboles
    # Media (6-15) → OrdinalEncoder o TargetEncoder
    # Alta (>15) → TargetEncoder o frecuencia
}
encoding_rows = []
for c in categoricas:
    n_cats = df[c].nunique()
    v_cramer = df_chi2[df_chi2['variable'] == c]['cramer_v'].values
    v = float(v_cramer[0]) if len(v_cramer) > 0 else 0
    if n_cats <= 2:
        enc_lineal = 'OrdinalEncoder (0/1)'
        enc_arbol  = 'Sin encoding (0/1)'
        just = 'Binaria — directamente usable'
    elif n_cats <= 5:
        enc_lineal = 'OneHotEncoder'
        enc_arbol  = 'OrdinalEncoder'
        just = f'Baja cardinalidad ({n_cats} cats) — OHE sin riesgo de dimensionalidad'
    elif n_cats <= 15:
        enc_lineal = 'OrdinalEncoder + TargetEncoder'
        enc_arbol  = 'OrdinalEncoder'
        just = f'Cardinalidad media ({n_cats} cats) — TargetEncoder si V Cramér > 0.1'
    else:
        enc_lineal = 'TargetEncoder'
        enc_arbol  = 'TargetEncoder'
        just = f'Alta cardinalidad ({n_cats} cats) — OHE genera demasiadas columnas'
    encoding_rows.append({
        'variable': c, 'etiqueta': etiq(c),
        'n_cats': n_cats, 'cramer_v': v,
        'enc_lineal': enc_lineal, 'enc_arbol': enc_arbol,
        'justificacion': just,
    })

df_encoding = pd.DataFrame(encoding_rows).sort_values('n_cats')

# --- Nulos detallado ---
nulos_cat = {c: df[c].isna().sum() for c in categoricas if df[c].isna().sum() > 0}
df_nulos_cat = pd.DataFrame([
    {'variable': c, 'etiqueta': etiq(c),
     'n_nulos': n, 'pct_nulos': round(n/len(df)*100, 1),
     'severidad': 'Alta' if n/len(df) > 0.3 else 'Media' if n/len(df) > 0.1 else 'Baja'}
    for c, n in sorted(nulos_cat.items(), key=lambda x: -x[1])
])

# --- Tasa de abandono por categoría (top diferencia) ---
tasa_por_cat = []
for c in categoricas:
    tasas = df.groupby(c)['abandono'].agg(['mean','count']).reset_index()
    tasas.columns = [c, 'tasa', 'n']
    tasas = tasas[tasas['n'] >= 20]
    for _, row in tasas.iterrows():
        tasa_por_cat.append({
            'variable': etiq(c), 'categoria': str(row[c])[:35],
            'tasa_pct': round(row['tasa']*100, 1), 'n': int(row['n'])
        })
df_tasas_cat = pd.DataFrame(tasa_por_cat).sort_values('tasa_pct', ascending=False)

# --- Resumen ejecutivo dinámico ---
top_chi2 = df_chi2.iloc[0]
var_mas_nulos = df_nulos_cat.iloc[0] if len(df_nulos_cat) > 0 else None
cat_mayor_abandono = df_tasas_cat.iloc[0] if len(df_tasas_cat) > 0 else None
n_significativas = df_chi2['significativa'].sum()

print(f"\nResumen ejecutivo generado")


Dataset: 33,621 filas × 26 columnas

Categóricas: 9
  Baja (≤5):   ['rama', 'sexo', 'universidad_origen']
  Media (6-15): ['cupo', 'situacion_laboral', 'via_acceso']
  Alta (>15):   ['pais_nombre', 'provincia', 'titulacion']
  Con nulos: 3

Resumen de dominancia:
         Variable  Categorías  Nulos  % Nulos                                Moda  % Moda  Entropía  Entropía máx.  Ratio entropía
             Sexo           2      0      0.0                               Mujer    55.1      0.99           1.00            0.99
             Rama           5      0      0.0                                  SO    55.3      1.74           2.32            0.75
     Univ. origen           5   4699     14.0                                 UJI    69.6      1.31           2.32            0.57
             cupo          10   4699     14.0                             General    96.3      0.29           3.32            0.09
Situación laboral          11  12740     37.9              Inactivo o desempleado

In [4]:
# ============================================================================
# CELDA 3: WAFFLE SEXO + CARDINALIDAD + DONUT UNIVERSIDAD + MAPA
# ============================================================================

graficos = {}

# --- 1. WAFFLE CHART: SEXO (PyWaffle) — 1 cuadrado = 10% ---
vc_sexo = df['sexo'].value_counts()
pct_mujer = round(vc_sexo['Mujer'] / n_total * 100)
pct_hombre = round(vc_sexo['Hombre'] / n_total * 100)
# Redondear a decenas para 10 cuadrados
bloques_mujer = round(pct_mujer / 10)
bloques_hombre = 10 - bloques_mujer

fig = plt.figure(
    FigureClass=Waffle,
    rows=2,
    columns=5,
    values=[bloques_mujer, bloques_hombre],
    colors=[COLOR_MUJER, COLOR_HOMBRE],
    labels=[f'Mujer: {fmt(vc_sexo["Mujer"])} ({pct_mujer}%)',
            f'Hombre: {fmt(vc_sexo["Hombre"])} ({pct_hombre}%)'],
    title={'label': f'Distribución por Sexo — {fmt(n_total)} estudiantes  (1 cuadrado = 10%)',
           'loc': 'center', 'fontsize': 12, 'fontweight': 'bold'},
    legend={'loc': 'lower center', 'ncol': 2, 'fontsize': 10,
            'bbox_to_anchor': (0.5, -0.15)},
    figsize=(6, 3),
    block_arranging_style='snake',
    interval_ratio_x=0.3,
    interval_ratio_y=0.3,
)
fig.subplots_adjust(left=0.15, right=0.85, top=0.85, bottom=0.15)

graficos['waffle_sexo'] = img_html(fig_a_base64(fig), ancho='50%')
print(f'✅ Waffle sexo (compacto 5×2, 1 cuadrado = 10%)')


# --- 2. CARDINALIDAD GENERAL (Altair) ---
df_card = pd.DataFrame({
    'Variable': [etiq(c) for c in categoricas],
    'Categorías': [df[c].nunique() for c in categoricas],
    'Grupo': ['🟢 Baja (≤5)' if df[c].nunique() <= 5
              else '🔵 Media (6-15)' if df[c].nunique() <= 15
              else '🟠 Alta (>15)' for c in categoricas],
    'Nulos': [df[c].isna().sum() for c in categoricas],
}).sort_values('Categorías', ascending=True)

chart_card = alt.Chart(df_card).mark_bar().encode(
    y=alt.Y('Variable:N', sort='-x', title=''),
    x=alt.X('Categorías:Q', title='Número de categorías'),
    color=alt.Color('Grupo:N',
                    scale=alt.Scale(
                        domain=['🟢 Baja (≤5)', '🔵 Media (6-15)', '🟠 Alta (>15)'],
                        range=[COLOR_OK, COLOR_PRINCIPAL, COLOR_ALERTA]),
                    legend=alt.Legend(title='Cardinalidad')),
    tooltip=['Variable', 'Categorías', 'Grupo', 'Nulos']
).properties(
    title='Cardinalidad de Variables Categóricas',
    width=550, height=300
).configure_title(fontSize=14, fontWeight='bold')

graficos['cardinalidad'] = guardar_altair(chart_card, 'm04_cardinalidad')
print('✅ Cardinalidad (Altair)')


# --- 3. DONUT: UNIVERSIDAD DE ORIGEN (Pygal SVG) ---
from pygal.style import Style

estilo_uni = Style(
    colors=('#e53e3e', '#3182ce', '#38a169', '#ed8936', '#805ad5'),
    font_family='Arial',
    title_font_size=16,
    label_font_size=11,
    value_font_size=11,
    tooltip_font_size=12,
    background='transparent',
)

vc_uni = df['universidad_origen'].value_counts()
n_con_dato = df['universidad_origen'].notna().sum()
n_sin_dato = df['universidad_origen'].isna().sum()

donut_uni = pygal.Pie(
    style=estilo_uni,
    inner_radius=0.55,
    title=f'Universidad de Origen (N={fmt(n_con_dato)}, sin dato={fmt(n_sin_dato)})',
    width=600,
    height=400,
    print_values=True,
    value_formatter=lambda x: f'{x:.1f}%',
)

for uni, count in vc_uni.items():
    pct = count / n_con_dato * 100
    donut_uni.add(f'{uni} ({fmt(count)})', round(pct, 1))

# Pygal SVG → inline
svg_uni = donut_uni.render().decode('utf-8')
graficos['donut_universidad'] = f'<div style="max-width:600px;margin:0 auto;">{svg_uni}</div>'
print('✅ Donut universidad origen (Pygal)')


# --- 4. MAPA: UNIVERSIDADES DE ORIGEN (Folium) — datos dinámicos desde el dataset ---
# Coordenadas fijas por universidad (dato geográfico estático, no del dataset)
COORDS_UNIS = {
    'UJI': {'nombre': 'UJI (Universitat Jaume I)',      'ciudad': 'Castellón', 'lat': 39.994444, 'lon': -0.068889, 'color': '#e53e3e'},
    'UV':  {'nombre': 'UV (Universitat de València)',    'ciudad': 'Valencia',  'lat': 39.467,    'lon': -0.377,    'color': '#3182ce'},
    'UPV': {'nombre': 'UPV (Universitat Politècnica)',  'ciudad': 'Valencia',  'lat': 39.483,    'lon': -0.320,    'color': '#38a169'},
    'UMH': {'nombre': 'UMH (Univ. Miguel Hernández)',   'ciudad': 'Elche',     'lat': 38.275000, 'lon': -0.684167, 'color': '#ed8936'},
    'UA':  {'nombre': 'UA (Universitat d\'Alacant)',    'ciudad': 'Alicante',  'lat': 38.384100, 'lon': -0.507900, 'color': '#805ad5'},
}

# Calcular alumnos y % desde el dataset — dinámico
vc_uni_mapa = df['universidad_origen'].value_counts()
n_uni_total = df['universidad_origen'].notna().sum()

unis = []
for codigo, info in COORDS_UNIS.items():
    # Buscar el código en los valores del dataset
    n_alumnos = int(vc_uni_mapa.get(codigo, 0))
    pct = round(n_alumnos / n_uni_total * 100, 1) if n_uni_total > 0 else 0
    if n_alumnos > 0:
        unis.append({**info, 'alumnos': n_alumnos, 'pct': pct})

print(f'Universidades en mapa: {len(unis)} con datos reales del dataset')

m = folium.Map(
    location=[39.1, -0.5],
    zoom_start=8,
    tiles='cartodbpositron'
)

for uni in unis:
    radio = max(6, math.sqrt(uni['alumnos']) * 0.35)
    
    folium.CircleMarker(
        location=[uni['lat'], uni['lon']],
        radius=radio,
        color=uni['color'],
        fill=True,
        fill_color=uni['color'],
        fill_opacity=0.7,
        weight=2,
        popup=folium.Popup(
            f"<div style='font-family:Arial;'>"
            f"<b style='font-size:13px;'>{uni['nombre']}</b><br>"
            f"📍 {uni['ciudad']}<br>"
            f"👥 {uni['alumnos']:,} alumnos<br>"
            f"📊 {uni['pct']}% del total</div>",
            max_width=280
        ),
        tooltip=f"{uni['nombre']}: {uni['alumnos']:,} ({uni['pct']}%)"
    ).add_to(m)

# Leyenda
# Leyenda dinámica desde datos reales
_filas_leyenda = ''.join([
    f'<span style="color:{u["color"]}">●</span> '
    f'{u["nombre"].split(" ")[0]} — {fmt(u["alumnos"])} ({u["pct"]}%)<br>'
    for u in sorted(unis, key=lambda x: -x['alumnos'])
])
legend_html = f"""
<div style="position:fixed;bottom:30px;left:30px;z-index:1000;
background:white;padding:10px;border-radius:8px;border:1px solid #ccc;
font-family:Arial;font-size:11px;box-shadow:0 2px 6px rgba(0,0,0,0.2);">
<b>Universidad de origen</b><br>
{_filas_leyenda}
<i>Tamaño ∝ nº alumnos</i>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

ruta_mapa = RUTA_GRAFICOS / 'm04_mapa_universidades.html'
m.save(str(ruta_mapa))
graficos['mapa_universidades'] = f'<iframe src="graficos/m04_mapa_universidades.html" width="100%" height="500" frameborder="0" style="border-radius:8px;"></iframe>'
print('✅ Mapa universidades (Folium)')


✅ Waffle sexo (compacto 5×2, 1 cuadrado = 10%)
✅ Cardinalidad (Altair)
✅ Donut universidad origen (Pygal)
Universidades en mapa: 5 con datos reales del dataset
✅ Mapa universidades (Folium)


In [5]:
# ============================================================================
# CELDA 4: GRÁFICOS — RAMA, DROPDOWN CATEGÓRICAS, ORDEN PREF, TABLAS
# ============================================================================

# --- 4. TREEMAP: RAMA (Plotly) ---
vc_rama = df['rama'].value_counts().reset_index()
vc_rama.columns = ['Rama', 'Alumnos']
nombres_rama = {
    'SO': 'Ciencias Sociales y Jurídicas',
    'TE': 'Ingeniería y Arquitectura',
    'SA': 'Ciencias de la Salud',
    'HU': 'Artes y Humanidades',
    'EX': 'Ciencias Experimentales',
}
vc_rama['Nombre'] = vc_rama['Rama'].map(nombres_rama)
vc_rama['Pct'] = (vc_rama['Alumnos'] / n_total * 100).round(1)
vc_rama['Label'] = vc_rama.apply(
    lambda r: f"{r['Nombre']}<br>{r['Rama']} — {fmt(r['Alumnos'])} ({r['Pct']}%)", axis=1)

fig_rama = px.treemap(
    vc_rama, path=['Label'], values='Alumnos',
    color='Alumnos',
    color_continuous_scale=['#ebf8ff', '#3182ce', '#2c5282'],
    title=f'Distribución por Rama de Conocimiento — {fmt(n_total)} estudiantes',
)
fig_rama.update_traces(
    textinfo='label+value', textfont_size=12,
    hovertemplate='<b>%{label}</b><br>Alumnos: %{value:,}<extra></extra>'
)
fig_rama.update_layout(
    margin=dict(t=40, l=10, r=10, b=10),
    coloraxis_showscale=False, title_font_size=14,
)
graficos['treemap_rama'] = '<div style="max-width:700px;margin:0 auto;">' + guardar_plotly(fig_rama, 'm04_treemap_rama', height=350) + '</div>'
print('✅ Treemap rama (Plotly)')


# --- 5. DROPDOWN UNIFICADO: 5 VARIABLES CATEGÓRICAS (Altair) ---
variables_drop = {
    'Situación laboral': ('situacion_laboral', 15),
    'Vía de acceso': ('via_acceso', 15),
    'Titulación': ('titulacion', 15),
    'Provincia': ('provincia', 15),
    'País': ('pais_nombre', 15),
}

rows_drop = []
for var_label, (col, top_n) in variables_drop.items():
    vc = df[col].value_counts()
    n_validos = df[col].notna().sum()
    n_nulos = df[col].isna().sum()
    
    top = vc.head(top_n)
    if len(vc) > top_n:
        otros = vc.iloc[top_n:].sum()
        cats = list(top.index) + [f'⋯ Otras ({len(vc)-top_n})']
        vals = list(top.values) + [otros]
    else:
        cats = list(top.index)
        vals = list(top.values)
    
    for cat, val in zip(cats, vals):
        pct = val / n_validos * 100
        rows_drop.append({
            'Variable': var_label,
            'Categoría': str(cat)[:45],
            'Alumnos': val,
            'Porcentaje': round(pct, 1),
            'Info': f'N={n_validos:,}, nulos={n_nulos:,} ({n_nulos/n_total*100:.1f}%)'
        })

df_drop = pd.DataFrame(rows_drop)

dropdown = alt.binding_select(
    options=list(variables_drop.keys()),
    name='📊 Variable categórica:  '
)
selection = alt.selection_point(
    fields=['Variable'],
    bind=dropdown,
    value='Situación laboral'
)

# Color distinto por variable
color_por_var = {
    'Situación laboral': '#3182ce',
    'Vía de acceso': '#38a169',
    'Titulación': '#805ad5',
    'Provincia': '#ed8936',
    'País': '#e53e3e',
}
df_drop['Color'] = df_drop['Variable'].map(color_por_var)

# Texto dinámico con nombre de variable (actúa como título)
titulo_dinamico = alt.Chart(df_drop).mark_text(
    fontSize=15, fontWeight='bold', align='center', dy=-5
).encode(
    text='Variable:N',
    color=alt.Color('Variable:N',
                    scale=alt.Scale(
                        domain=list(color_por_var.keys()),
                        range=list(color_por_var.values())),
                    legend=None),
).transform_filter(
    selection
).transform_aggregate(
    groupby=['Variable']
).properties(
    width=580, height=20
)

# Barras con valores encima y color por variable
barras = alt.Chart(df_drop).mark_bar().encode(
    y=alt.Y('Categoría:N', sort='-x', title=''),
    x=alt.X('Alumnos:Q', title='Nº de estudiantes'),
    color=alt.Color('Variable:N',
                    scale=alt.Scale(
                        domain=list(color_por_var.keys()),
                        range=list(color_por_var.values())),
                    legend=None),
    tooltip=[
        alt.Tooltip('Categoría:N'),
        alt.Tooltip('Alumnos:Q', format=','),
        alt.Tooltip('Porcentaje:Q', title='%', format='.1f'),
        alt.Tooltip('Info:N', title='Datos'),
    ]
).transform_filter(
    selection
).properties(
    width=580, height=400
)

# Etiquetas con % en cada barra
etiquetas = alt.Chart(df_drop).mark_text(
    align='left', dx=4, fontSize=10, color='#4a5568'
).encode(
    y=alt.Y('Categoría:N', sort='-x'),
    x=alt.X('Alumnos:Q'),
    text=alt.Text('Porcentaje:Q', format='.1f'),
).transform_filter(
    selection
)

chart_drop = alt.vconcat(
    titulo_dinamico,
    (barras + etiquetas)
).add_params(
    selection
).resolve_scale(
    color='shared'
)

graficos['dropdown_categoricas'] = guardar_altair(chart_drop, 'm04_dropdown_categoricas')
print('✅ Dropdown 5 variables categóricas (Altair)')


# --- 6. BARRAS: ORDEN DE PREFERENCIA (Plotly degradado) ---
vc_ord = df['orden_preferencia'].dropna().astype(int).value_counts().sort_index()
n_ord_validos = df['orden_preferencia'].notna().sum()
n_ord_nulos = df['orden_preferencia'].isna().sum()

fig_ord = go.Figure(go.Bar(
    x=vc_ord.index, y=vc_ord.values,
    marker=dict(
        color=vc_ord.values,
        colorscale=[[0, '#ebf8ff'], [0.3, '#63b3ed'], [0.6, '#3182ce'], [1, '#1a365d']],
        colorbar=dict(title='Alumnos'),
    ),
    text=[f'{v/n_ord_validos*100:.1f}%' for v in vc_ord.values],
    textposition='outside',
    hovertemplate='Opción %{x}<br>Alumnos: %{y:,}<br>%{text}<extra></extra>'
))
fig_ord.update_layout(
    title=dict(text=f'Orden de Preferencia 1-20 (N={fmt(n_ord_validos)}, sin dato={fmt(n_ord_nulos)})',
               font_size=14),
    xaxis_title='Orden de preferencia',
    yaxis_title='Nº de estudiantes',
    xaxis=dict(dtick=1),
    margin=dict(t=50, b=40),
    template='plotly_white',
)
graficos['orden_preferencia'] = guardar_plotly(fig_ord, 'm04_orden_preferencia', height=450)
print('✅ Orden preferencia (Plotly)')


# --- 7. TABLA DE DOMINANCIA Y ENTROPÍA ---
tabla_dom = df_resumen.to_html(index=False, border=1,
    classes='tabla-estadisticas',
    float_format=lambda x: f'{x:.2f}' if isinstance(x, float) else x)
graficos['tabla_dominancia'] = (
    '<div style="overflow-x:auto;max-height:400px;overflow-y:auto;">'
    + tabla_dom + '</div>'
)
print('✅ Tabla dominancia + entropía')


# --- 8. CATEGORÍAS RARAS < 50 casos ---
raras = []
for c in categoricas:
    vc = df[c].value_counts()
    cats_raras = vc[vc < 50]
    for cat, count in cats_raras.items():
        raras.append({'Variable': etiq(c), 'Categoría': str(cat)[:40],
                      'N': count, '% del total': round(count/n_total*100, 3)})

df_raras = pd.DataFrame(raras).sort_values('N')
tabla_raras = df_raras.to_html(index=False, border=1, classes='tabla-estadisticas')
graficos['tabla_raras'] = (
    '<div style="overflow-x:auto;max-height:350px;overflow-y:auto;">'
    + tabla_raras + '</div>'
)
print(f'✅ Categorías raras: {len(raras)} categorías con < 50 casos')


✅ Treemap rama (Plotly)
✅ Dropdown 5 variables categóricas (Altair)
✅ Orden preferencia (Plotly)
✅ Tabla dominancia + entropía
✅ Categorías raras: 119 categorías con < 50 casos


✅ Dropdown 5 variables categóricas (Altair)
✅ Orden preferencia (Plotly)
✅ Tabla dominancia + entropía
✅ Categorías raras: 112 categorías con < 50 casos


In [6]:
# ============================================================================
# CELDA 5: GENERAR HTML
# ============================================================================


# ── Variables dinámicas para textos del HTML ─────────────────────────────────
_pct_mujer = round(df['sexo'].value_counts(normalize=True).get('Mujer', 0) * 100, 1)
_pct_hombre = round(100 - _pct_mujer, 1)
_rama_dom = df['rama'].value_counts().index[0]
_rama_dom_nombre = {'SO': 'Ciencias Sociales', 'TE': 'Ingeniería y Arquitectura',
    'SA': 'Ciencias de la Salud', 'HU': 'Artes y Humanidades',
    'EX': 'Ciencias Experimentales'}.get(_rama_dom, _rama_dom)
_rama_dom_pct = round(df['rama'].value_counts(normalize=True).iloc[0] * 100, 1)
_rama_min = df['rama'].value_counts().index[-1]
_rama_min_nombre = {'SO': 'Ciencias Sociales', 'TE': 'Ingeniería y Arquitectura',
    'SA': 'Ciencias de la Salud', 'HU': 'Artes y Humanidades',
    'EX': 'Ciencias Experimentales'}.get(_rama_min, _rama_min)
_rama_min_pct = round(df['rama'].value_counts(normalize=True).iloc[-1] * 100, 1)
_pct_via_principal = round(df['via_acceso'].value_counts(normalize=True).iloc[0] * 100, 1)
_via_principal = df['via_acceso'].value_counts().index[0]
_pct_pais_dom = round(df['pais_nombre'].value_counts(normalize=True).iloc[0] * 100, 1)
_pais_dom = df['pais_nombre'].value_counts().index[0]
_pct_ord1 = round((df['orden_preferencia'] == 1).sum() / df['orden_preferencia'].notna().sum() * 100, 1)
_pct_sit_nulos = round(df['situacion_laboral'].isna().mean() * 100, 1)
_entropia_pais = df_resumen[df_resumen['Variable'] == etiq('pais_nombre')]['Ratio entropía'].values
_entropia_pais = round(float(_entropia_pais[0]), 2) if len(_entropia_pais) > 0 else 0
_entropia_sexo = df_resumen[df_resumen['Variable'] == etiq('sexo')]['Ratio entropía'].values
_entropia_sexo = round(float(_entropia_sexo[0]), 2) if len(_entropia_sexo) > 0 else 0


# ── Resumen ejecutivo dinámico ────────────────────────────────────────────────
resumen_ejecutivo_html = (
    '<div style="background:linear-gradient(135deg,#ebf8ff,#f0fff4);padding:20px;'
    'border-radius:10px;border-left:4px solid #3182ce;margin-bottom:20px;">'
    f'<h3 style="margin:0 0 12px;color:#1a202c;">📋 Resumen ejecutivo — {len(categoricas)} variables categóricas</h3>'
    f'<p>• <strong>Variable más predictiva:</strong> {top_chi2["etiqueta"]} '
    f'(V Cramér = {top_chi2["cramer_v"]:.3f}, {top_chi2["magnitud"]}) — '
    f'rango de tasas de abandono de {top_chi2["rango_tasas"]:.1f} pp.</p>'
    + (f'<p>• <strong>Mayor % de nulos:</strong> {var_mas_nulos["etiqueta"]} '
       f'({var_mas_nulos["pct_nulos"]}% sin dato — severidad {var_mas_nulos["severidad"]}).</p>'
       if var_mas_nulos is not None else '')
    + (f'<p>• <strong>Categoría con mayor abandono:</strong> '
       f'{cat_mayor_abandono["categoria"]} en {cat_mayor_abandono["variable"]} '
       f'({cat_mayor_abandono["tasa_pct"]}%, n={fmt(cat_mayor_abandono["n"])}).</p>'
       if cat_mayor_abandono is not None else '')
    + f'<p>• <strong>Significativas (p&lt;0.05):</strong> {n_significativas} de {len(categoricas)} variables.</p>'
    + '<p style="font-size:0.85em;color:#4a5568;margin-top:8px;">'
    'Todas las variables entran a Fase 5. Encoding y tratamiento de nulos '
    'documentados en las secciones siguientes.</p>'
    '</div>'
)

# ── Tabla Chi² + V Cramér ─────────────────────────────────────────────────────
COLOR_MAG = {'Fuerte': '#fed7d7', 'Moderada': '#fefcbf', 'Débil': '#f0fff4'}
_COLOR_MAG = {'Fuerte': '#e53e3e', 'Moderada': '#d69e2e', 'Débil': '#38a169'}
filas_chi2 = ''
for _, r in df_chi2.iterrows():
    bg = COLOR_MAG.get(r['magnitud'], 'white')
    sig = '✅' if r['significativa'] else '⚠️'
    filas_chi2 += (
        f'<tr style="background:{bg};">'
        f'<td style="padding:8px;font-weight:600;">{r["etiqueta"]}</td>'
        f'<td style="text-align:center;">{r["cramer_v"]:.4f}</td>'
        f'<td style="text-align:center;">'
        f'<span style="background:{_COLOR_MAG.get(r["magnitud"],"#718096")};'
        f'color:white;padding:2px 8px;border-radius:4px;font-size:0.85em;">{r["magnitud"]}</span></td>'
        f'<td style="text-align:center;">{r["rango_tasas"]:.1f} pp</td>'
        f'<td style="text-align:center;">{sig}</td>'
        f'</tr>'
    )

tabla_chi2_html = (
    '<table style="width:100%;border-collapse:collapse;font-size:0.9em;">'
    '<thead><tr style="background:#2d3748;color:white;">'
    '<th style="padding:10px;text-align:left;">Variable</th>'
    '<th style="padding:10px;text-align:center;">V Cramér</th>'
    '<th style="padding:10px;text-align:center;">Magnitud</th>'
    '<th style="padding:10px;text-align:center;">Rango tasas</th>'
    '<th style="padding:10px;text-align:center;">Sig.</th>'
    '</tr></thead>'
    f'<tbody>{filas_chi2}</tbody>'
    '</table>'
    '<div style="background:#ebf8ff;padding:8px;border-radius:6px;margin-top:8px;font-size:0.85em;">'
    'V Cramér: &gt;0.3 Fuerte · 0.1–0.3 Moderada · &lt;0.1 Débil. '
    'Con 33.621 registros casi todo es significativo — la V de Cramér es el indicador real.'
    '</div>'
)
s_chi2 = generar_seccion_html(
    '📊 Asociación con Abandono — Chi² + V de Cramér',
    h2btn('Asociación con Abandono', 'chi2') + panel_chi2 + tabla_chi2_html
)

# ── Tabla encoding ────────────────────────────────────────────────────────────
COLOR_ENC = {'OHE': '#ebf8ff', 'Ord': '#f0fff4', 'Target': '#fffbeb', 'Sin': '#f0fff4'}
filas_enc = ''
for _, r in df_encoding.iterrows():
    filas_enc += (
        f'<tr>'
        f'<td style="padding:8px;font-weight:600;">{r["etiqueta"]}</td>'
        f'<td style="text-align:center;">{r["n_cats"]}</td>'
        f'<td style="text-align:center;">{r["cramer_v"]:.3f}</td>'
        f'<td style="padding:8px;font-size:0.85em;">'
        f'<span style="background:#3182ce;color:white;padding:1px 6px;border-radius:3px;font-size:0.8em;">'
        f'Lineal</span> {r["enc_lineal"]}</td>'
        f'<td style="padding:8px;font-size:0.85em;">'
        f'<span style="background:#38a169;color:white;padding:1px 6px;border-radius:3px;font-size:0.8em;">'
        f'Árbol</span> {r["enc_arbol"]}</td>'
        f'<td style="padding:8px;font-size:0.82em;color:#4a5568;">{r["justificacion"]}</td>'
        f'</tr>'
    )

tabla_enc_html = (
    '<table style="width:100%;border-collapse:collapse;font-size:0.9em;">'
    '<thead><tr style="background:#2d3748;color:white;">'
    '<th style="padding:10px;text-align:left;">Variable</th>'
    '<th style="padding:10px;text-align:center;">Cats.</th>'
    '<th style="padding:10px;text-align:center;">V Cramér</th>'
    '<th style="padding:10px;">Modelos lineales</th>'
    '<th style="padding:10px;">Modelos árbol</th>'
    '<th style="padding:10px;">Justificación</th>'
    '</tr></thead>'
    f'<tbody>{filas_enc}</tbody>'
    '</table>'
    '<div style="background:#f0f4f8;padding:8px;border-radius:6px;margin-top:8px;font-size:0.85em;">'
    'Esta tabla es la fuente de decisión para el pipeline de encoding de Fase 5. '
    'TargetEncoder requiere cross-validation para evitar leakage.'
    '</div>'
)
s_encoding = generar_seccion_html(
    '🏷️ Encoding Recomendado para Fase 5 — Semáforo por Variable',
    tabla_enc_html
)

# ── Gráfico nulos categóricas ─────────────────────────────────────────────────
if len(df_nulos_cat) > 0:
    import matplotlib
    import matplotlib.pyplot as plt
    import io, base64
    fig_nulos, ax_n = plt.subplots(figsize=(8, max(3, len(df_nulos_cat) * 0.7)))
    colors_n = [COLOR_ERROR if r['pct_nulos'] > 30 else COLOR_ALERTA
                for _, r in df_nulos_cat.iterrows()]
    bars_n = ax_n.barh(df_nulos_cat['etiqueta'], df_nulos_cat['pct_nulos'],
                       color=colors_n, alpha=0.85, edgecolor='white')
    for bar, (_, row) in zip(bars_n, df_nulos_cat.iterrows()):
        ax_n.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                  f"{fmt(row['n_nulos'])} ({row['pct_nulos']}%) — {row['severidad']}",
                  va='center', fontsize=9)
    ax_n.set_title('Valores Faltantes en Variables Categóricas',
                   fontsize=13, fontweight='bold')
    ax_n.set_xlabel('% valores faltantes')
    ax_n.set_xlim(0, df_nulos_cat['pct_nulos'].max() * 1.35)
    fig_nulos.tight_layout()
    buf_n = io.BytesIO()
    fig_nulos.savefig(buf_n, format='png', dpi=150, bbox_inches='tight', facecolor='white')
    plt.close(fig_nulos)
    buf_n.seek(0)
    b64_n = base64.b64encode(buf_n.read()).decode('utf-8')
    graf_nulos_cat = f'<img src="data:image/png;base64,{b64_n}" style="max-width:100%;border-radius:8px;">'
    s_nulos_cat = generar_seccion_html(
        '⚠️ Valores Faltantes en Variables Categóricas',
        graf_nulos_cat
    )
else:
    s_nulos_cat = generar_seccion_html(
        '⚠️ Valores Faltantes en Variables Categóricas',
        '<p style="color:#38a169;">✅ No hay valores faltantes en variables categóricas.</p>'
    )


# ── Función para nombres legibles en gráficos ────────────────────────────────
def limpiar_nombre_cat(col, val):
    s = str(val)
    if col == 'cupo':
        import re
        s = re.sub(r'(\d)(años|Años)', r'\1 \2', s)
        return s
    if col == 'rama':
        return DICCIONARIO_RAMAS.get(s, s)
    if col == 'universidad_origen':
        return UNIVERSIDAD_ORIGEN_NOMBRES.get(s, s)
    if col == 'titulacion':
        s = s.replace('Grado en ', 'G. ').replace('Doble Grado en ', 'D.G. ')
        s = s.replace('Ingeniería ', 'Ing. ').replace('Administración ', 'Adm. ')
        return s[:40]
    # Limpiar espacios y formato incorrecto
    s = s.strip()
    return s[:35]

# ── Tasa de abandono por categoría — un gráfico por variable ────────────────
tasa_global_pct = df['abandono'].mean() * 100
graficos_tasas = []  # lista de HTML por variable

for c in categoricas:
    tasas_c = df.groupby(c)['abandono'].agg(['mean','count']).reset_index()
    tasas_c.columns = [c, 'tasa', 'n']
    tasas_c = tasas_c[tasas_c['n'] >= 10].sort_values('tasa', ascending=True)
    tasas_c['tasa_pct'] = (tasas_c['tasa'] * 100).round(1)

    # Nombres legibles sin truncar
    cats_label = [limpiar_nombre_cat(c, v) for v in tasas_c[c]]
    colors_t = ['#e53e3e' if t > tasa_global_pct else '#3182ce'
                for t in tasas_c['tasa_pct']]

    # Altura dinámica según número de categorías
    altura = max(300, len(tasas_c) * 38)
    # Margen izquierdo según longitud máxima de etiqueta
    max_len = max((len(l) for l in cats_label), default=10)
    margen_izq = max(120, min(280, max_len * 7))

    fig_c = go.Figure(go.Bar(
        y=cats_label, x=tasas_c['tasa_pct'],
        orientation='h',
        marker_color=colors_t,
        text=[f"{t}%" for t in tasas_c['tasa_pct']],
        textposition='outside',
        cliponaxis=False,
        hovertemplate='<b>%{y}</b><br>Abandono: %{x:.1f}%<br>n=%{customdata:,}<extra></extra>',
        customdata=tasas_c['n'],
    ))
    fig_c.add_vline(
        x=tasa_global_pct, line_dash='dash',
        line_color='#e53e3e', line_width=1.5, opacity=0.6,
        annotation_text=f'Media: {tasa_global_pct:.1f}%',
        annotation_font=dict(size=10, color='#e53e3e'),
        annotation_position='top right'
    )
    fig_c.update_layout(
        title=dict(text=etiq(c), font=dict(size=13, color='#1a202c'), x=0.5),
        height=altura,
        template='plotly_white',
        margin=dict(t=50, b=30, l=margen_izq, r=80),
        xaxis=dict(title='% abandono', range=[0, max(tasas_c['tasa_pct'].max(), tasa_global_pct) * 1.25]),
        yaxis=dict(tickfont=dict(size=11)),
        font=dict(size=11),
    )
    graficos_tasas.append(fig_c.to_html(
        full_html=False, include_plotlyjs=False,
        config={'displayModeBar': False}
    ))

# Unir todos los gráficos
# Añadir Plotly JS una sola vez al inicio
_plotly_cdn = '<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>'
graf_tasas_html = _plotly_cdn + ''.join([
    f'<div style="margin-bottom:24px;">{g}</div>'
    for g in graficos_tasas
])

s_tasas_cat = generar_seccion_html(
    '📊 Tasa de Abandono por Categoría',
    graf_tasas_html
    + '<div style="background:#ebf8ff;padding:8px;border-radius:6px;'
    'margin-top:8px;font-size:0.85em;">'
    f'Rojo = por encima de la media global ({tasa_global_pct:.1f}%). '
    'Azul = por debajo. Solo categorías con n≥10.</div>'
)

# ── Categorías raras con acción sugerida ─────────────────────────────────────
raras_accion = []
for c in categoricas:
    vc = df[c].value_counts()
    tasa_c = df.groupby(c)['abandono'].mean()
    tasa_media = df['abandono'].mean()
    for cat, count in vc[vc < 50].items():
        tasa_cat = tasa_c.get(cat, tasa_media)
        diff = abs(tasa_cat - tasa_media) * 100
        if diff > 10:
            accion = f'Mantener — abandono {tasa_cat*100:.1f}% (diferencia {diff:.1f}pp)'
            color_acc = '#38a169'
        elif count < 10:
            accion = 'Eliminar — menos de 10 casos'
            color_acc = '#e53e3e'
        else:
            accion = 'Agrupar en "Otros"'
            color_acc = '#d69e2e'
        raras_accion.append({
            'Variable': etiq(c), 'Categoría': str(cat)[:40],
            'N': count, '%': round(count/len(df)*100, 3),
            'Abandono': f'{tasa_cat*100:.1f}%',
            'Acción': accion, '_color': color_acc
        })

if raras_accion:
    df_raras_acc = pd.DataFrame(raras_accion).sort_values('N')
    filas_raras = ''
    for _, r in df_raras_acc.iterrows():
        filas_raras += (
            f'<tr><td style="padding:6px;">{r["Variable"]}</td>'
            f'<td style="padding:6px;">{r["Categoría"]}</td>'
            f'<td style="text-align:center;">{fmt(r["N"])}</td>'
            f'<td style="text-align:center;">{r["%"]}%</td>'
            f'<td style="text-align:center;">{r["Abandono"]}</td>'
            f'<td style="padding:6px;">'
            f'<span style="background:{r["_color"]};color:white;padding:2px 6px;'
            f'border-radius:4px;font-size:0.82em;">{r["Acción"]}</span></td>'
            f'</tr>'
        )
    tabla_raras_acc = (
        '<table style="width:100%;border-collapse:collapse;font-size:0.88em;">'
        '<thead><tr style="background:#2d3748;color:white;">'
        '<th style="padding:8px;">Variable</th><th>Categoría</th>'
        '<th style="text-align:center;">N</th><th style="text-align:center;">%</th>'
        '<th style="text-align:center;">Abandono</th><th>Acción sugerida</th>'
        '</tr></thead>'
        f'<tbody>{filas_raras}</tbody></table>'
    )
    s_raras_acc = generar_seccion_html(
        '⚠️ Categorías Raras (<50 casos) — Acción Sugerida',
        tabla_raras_acc
        + '<div style="background:#fff5f5;padding:8px;border-radius:6px;'
        'margin-top:8px;font-size:0.85em;">'
        '🟢 Mantener si tasa de abandono muy diferente a la media. '
        '🟡 Agrupar en "Otros" si pocas observaciones. '
        '🔴 Eliminar si menos de 10 casos.</div>'
    )
else:
    s_raras_acc = ''


nav_fases, nav_modulos = generar_html_navegacion_completa(
    fase_activa='fase4', modulo_activo='m04'
)

# KPIs
kpis_html = generar_kpis_html([
    {'valor': str(len(categoricas)), 'titulo': 'Variables categóricas',
     'color': COLOR_PRINCIPAL},
    {'valor': str(len(baja_card)), 'titulo': 'Baja (≤5)',
     'color': COLOR_OK},
    {'valor': str(len(media_card)), 'titulo': 'Media (6-15)',
     'color': COLOR_ALERTA},
    {'valor': str(len(alta_card)), 'titulo': 'Alta (>15)',
     'color': COLOR_ERROR},
    {'valor': str(n_con_nulos), 'titulo': 'Con nulos',
     'color': '#805ad5'},
])

nota_librerias = (
    '<div style="background:#f0f4f8;padding:12px;border-radius:8px;'
    'border-left:4px solid #3182ce;margin-bottom:15px;font-size:0.85em;">'
    '<strong>📚 Librerías utilizadas:</strong> '
    'PyWaffle (pictograma sexo), '
    'Altair (dropdown interactivo 5 variables), '
    'Pygal (donut SVG), '
    'Plotly (treemap rama, barras orden preferencia), '
    'Folium (mapa geográfico). '
    'Interactivos como iframe, estáticos como PNG/SVG.</div>'
)

modal_css = """
<style>
.modal-overlay {
    display: none; position: fixed; top: 0; left: 0;
    width: 100%; height: 100%;
    background: rgba(0,0,0,0.7); z-index: 1000;
    justify-content: center; align-items: center;
}
.modal-content {
    background: white; border-radius: 10px;
    width: 90%; max-width: 1000px; max-height: 90%;
    overflow: auto; padding: 20px;
}
.modal-close {
    float: right; background: none; border: none;
    font-size: 1.5em; cursor: pointer; color: #718096;
}
.modal-close:hover { color: #e53e3e; }
.btn-mapa {
    display: inline-block; margin-top: 10px; padding: 10px 20px;
    background: #3182ce; color: white; border: none;
    border-radius: 8px; font-weight: 600; cursor: pointer; font-size: 0.9em;
}
.btn-mapa:hover { background: #2c5282; }
</style>
"""

# --- SECCIONES ---
s_cardinalidad = generar_seccion_html(
    'Cardinalidad de Variables Categóricas (Altair)',
    graficos['cardinalidad']
    + '<div style="background:#ebf8ff;padding:8px;border-radius:8px;'
    'border-left:4px solid #3182ce;margin-top:8px;font-size:0.85em;">'
    '<strong>Lectura:</strong> Pasa el ratón para ver categorías y nulos.</div>'
)

s_sexo = generar_seccion_html(
    'Distribución por Sexo (PyWaffle — Pictograma)',
    graficos['waffle_sexo']
    + '<div style="background:#ebf8ff;padding:8px;border-radius:8px;'
    'border-left:4px solid #3182ce;margin-top:8px;font-size:0.85em;">'
    f'<strong>Lectura:</strong> Cada cuadrado = 10%. '
    f'{_pct_mujer}% mujeres / {_pct_hombre}% hombres.</div>'
)

s_rama = generar_seccion_html(
    'Rama de Conocimiento (Plotly — Treemap)',
    graficos['treemap_rama']
    + '<div style="background:#ebf8ff;padding:8px;border-radius:8px;'
    'border-left:4px solid #3182ce;margin-top:8px;font-size:0.85em;">'
    f'<strong>Lectura:</strong> {_rama_dom_nombre} ({_rama_dom}) domina con {_rama_dom_pct}%. '
    f'{_rama_min_nombre} ({_rama_min}) tiene {_rama_min_pct}% — grupo minoritario a vigilar.</div>'
)

s_universidad = generar_seccion_html(
    'Universidad de Origen (Pygal donut + Folium mapa)',
    modal_css
    + graficos['donut_universidad']
    + '<div style="text-align:center;margin-top:10px;">'
    '<button class="btn-mapa" onclick="document.getElementById(\'modal-mapa\').style.display=\'flex\'">'
    '🗺️ Ver en mapa de la Comunitat Valenciana</button></div>'
    + '<div id="modal-mapa" class="modal-overlay" onclick="if(event.target==this)this.style.display=\'none\'">'
    '<div class="modal-content">'
    '<button class="modal-close" onclick="document.getElementById(\'modal-mapa\').style.display=\'none\'">×</button>'
    '<h3 style="margin-top:0;">Universidades de Origen — Comunitat Valenciana</h3>'
    + graficos['mapa_universidades']
    + '</div></div>'
    + '<div style="background:#ebf8ff;padding:8px;border-radius:8px;'
    'border-left:4px solid #3182ce;margin-top:8px;font-size:0.85em;">'
    '<strong>Lectura:</strong> 69,6% de la propia UJI. '
    'Las 5 universidades son de la Comunitat Valenciana. '
    '14% sin dato.</div>'
)

s_dropdown = generar_seccion_html(
    'Variables Categóricas — Explorador Interactivo (Altair)',
    '<div style="background:#f0f4f8;padding:10px;border-radius:8px;margin-bottom:8px;text-align:center;font-size:1em;">' 
    + '<strong>⬇️ Usa el desplegable bajo el gráfico para cambiar de variable</strong></div>'
    + graficos['dropdown_categoricas']
    + '<div style="background:#ebf8ff;padding:8px;border-radius:8px;'
    'border-left:4px solid #3182ce;margin-top:8px;font-size:0.85em;">'
    '<strong>Uso:</strong> Selecciona la variable en el desplegable superior '
    'para cambiar el gráfico. Pasa el ratón para ver detalles. '
    'Incluye: situación laboral (37,9% nulos), vía de acceso, '
    'titulación (top 15 de 40), provincia (top 15 de 50), '
    'país (top 15 de 77).<br><br>'
    '<strong>Hallazgos clave:</strong><br>'
    f'• <strong>Situación laboral:</strong> {_pct_sit_nulos}% sin dato — valorar inclusión en modelo.<br>'
    f'• <strong>Vía acceso:</strong> {_pct_via_principal}% por {_via_principal}.<br>'
    '• <strong>Titulación:</strong> 40 grados, ninguno supera el 7%. Muy repartida.<br>'
    '• <strong>Provincia:</strong> Castellón 70,8%, Valencia 23,9%.<br>'
    f'• <strong>País:</strong> {_pct_pais_dom}% {_pais_dom}.</div>'
)

s_orden = generar_seccion_html(
    'Orden de Preferencia en Preinscripción (Plotly)',
    graficos['orden_preferencia']
    + '<div style="background:#ebf8ff;padding:8px;border-radius:8px;'
    'border-left:4px solid #3182ce;margin-top:8px;font-size:0.85em;">'
    f'<strong>Lectura:</strong> {_pct_ord1}% eligió la UJI como 1ª opción. '
    'A partir de la 5ª opción los casos son residuales. '
    'Variable clave para M05: ¿abandonan más los que no eligieron la UJI primero?</div>'
)

s_dominancia = generar_seccion_html(
    'Dominancia y Entropía por Variable',
    graficos['tabla_dominancia']
    + '<div style="background:#ebf8ff;padding:8px;border-radius:8px;'
    'border-left:4px solid #3182ce;margin-top:8px;font-size:0.85em;">'
    f'<strong>Lectura:</strong> Ratio entropía (0 a 1): '
    '1 = categorías equilibradas, 0 = una domina. '
    f'{etiq("pais_nombre")} ({_entropia_pais}) = muy concentrado. '
    f'{etiq("sexo")} ({_entropia_sexo}) = muy equilibrado.</div>'
)

s_raras = generar_seccion_html(
    '⚠️ Categorías Raras (&lt;50 casos)',
    graficos['tabla_raras']
    + '<div style="background:#fff5f5;padding:8px;border-radius:8px;'
    'border-left:4px solid #e53e3e;margin-top:8px;font-size:0.85em;">'
    '<strong>Atención:</strong> Estas categorías tienen menos de 50 casos. '
    'Para el modelado, agrupar en "Otros" o eliminar para evitar sobreajuste.</div>'
)

contenido = (
    resumen_ejecutivo_html
    + kpis_html + nota_librerias
    + s_chi2
    + s_encoding
    + s_nulos_cat
    + s_tasas_cat
    + s_cardinalidad + s_sexo + s_rama + s_universidad
    + s_dropdown + s_orden + s_dominancia
    + s_raras_acc
)

html = render_base_html(
    titulo='M04: Distribuciones Categóricas',
    icono='\U0001f4ca',
    subtitulo='Fase 4: EDA Final | TFM Abandono Universitario',
    nav_fases=nav_fases,
    nav_modulos=nav_modulos,
    contenido=contenido,
    notebook_nombre='f4_m04_distribuciones_cat.ipynb',
    notebook_carpeta='fase4_eda',
)

ruta_html = RUTA_FASE4_HTML / 'm04_distribuciones_cat.html'
guardar_html(html, ruta_html)
print(f'\n✅ HTML generado: {ruta_html}')


✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase4\m04_distribuciones_cat.html

✅ HTML generado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase4\m04_distribuciones_cat.html


In [7]:
# ============================================================================
# CELDA 6: RESUMEN
# ============================================================================

print('=' * 70)
print('F4-M04: DISTRIBUCIONES CATEGÓRICAS — RESUMEN')
print('=' * 70)
print(f'  Dataset: {n_total:,} filas')
print(f'  Variables categóricas: {len(categoricas)}')
print(f'    - Baja cardinalidad:  {len(baja_card)} → {baja_card}')
print(f'    - Media cardinalidad: {len(media_card)} → {media_card}')
print(f'    - Alta cardinalidad:  {len(alta_card)} → {alta_card}')
print(f'  Con nulos: {n_con_nulos}')
print(f'  Gráficos generados: {len(graficos)}')
print(f'  Librerías: PyWaffle, Altair, Pygal, Plotly, Folium')
print(f'  HTML: {ruta_html}')
print('=' * 70)


F4-M04: DISTRIBUCIONES CATEGÓRICAS — RESUMEN
  Dataset: 33,621 filas
  Variables categóricas: 9
    - Baja cardinalidad:  3 → ['rama', 'sexo', 'universidad_origen']
    - Media cardinalidad: 3 → ['cupo', 'situacion_laboral', 'via_acceso']
    - Alta cardinalidad:  3 → ['pais_nombre', 'provincia', 'titulacion']
  Con nulos: 3
  Gráficos generados: 9
  Librerías: PyWaffle, Altair, Pygal, Plotly, Folium
  HTML: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase4\m04_distribuciones_cat.html
